<a href="https://colab.research.google.com/github/ja2faded/ds2002-fa26/blob/main/2026_09_11_%E2%80%94_SQL_Challenge_Set_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q1= q('''
SELECT tracks.title, artists.name, artists.country
FROM tracks
JOIN artists ON tracks.artist_id = artists.artist_id

''')
q1

,title,name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Foothills,The Blue Ridge,US
3,Aurora,Kestrel,UK
4,Nightfall,Kestrel,UK
5,Sol,Marisol,ES
6,Coastline,The Blue Ridge,US
7,Ridgeline,The Blue Ridge,US
8,Untitled Demo,Kestrel,UK


### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q2 =q('''
SELECT genre, AVG(seconds)
FROM tracks
GROUP BY genre
ORDER BY AVG(seconds) DESC
LIMIT 1

''')
q2

,genre,AVG(seconds)
0,Electronic,287.5


Electronic is the genre with the longest average track length. AVG(seconds)gets the average seconds. I ordered them descending to put the highest average seconds in the first row. LIMIT 1 makes it so only the first row is output.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q3 = q('''
SELECT user, COUNT(play_id) AS plays, COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
''')

q3

,user,plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


plays = disctinct tracks for every user

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q4 = q('''
SELECT tracks.track_id, tracks.title, plays.play_id AS plays
FROM tracks
LEFT JOIN plays ON tracks.track_id = plays.track_id
WHERE plays.play_id IS NULL

''')
q4

,track_id,title,plays
0,17,Ridgeline,None
1,18,Untitled Demo,None


LEFT JOIN keeps all rows from tracks

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q5 = q('''
SELECT artists.name, SUM(tracks.seconds), ROUND(SUM(tracks.seconds)/60.0, 1) AS minutes
FROM artists
JOIN tracks ON artists.artist_id = tracks.artist_id
JOIN plays ON tracks.track_id = plays.track_id
GROUP BY artists.name
ORDER BY minutes DESC
''')
q5

,name,SUM(tracks.seconds),minutes
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


Chaining artists JOIN tracks JOIN plays makes each output row one play event. SUM(tracks.seconds) adds all listening time (repeats included).

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [7]:
q6= q('''
SELECT track_id, title
FROM tracks
WHERE genre is NULL

''')
q6

,track_id,title
0,18,Untitled Demo


In [8]:
q('''
SELECT track_id, title
FROM tracks
WHERE genre != 'Pop'

''')


,track_id,title
0,12,Foothills
1,13,Aurora
2,14,Nightfall
3,15,Sol
4,16,Coastline
5,17,Ridgeline


WHERE genre != 'Pop' returns all tracks whose genre was not Pop. The track (Untitled Demo) with no genre (NULL) is excluded from this.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [9]:
q7 = q('''
SELECT played_on, COUNT(play_id) AS plays, COUNT(DISTINCT user) AS distinct_users
FROM plays
GROUP BY played_on
ORDER BY played_on ASC

''')

q7

,played_on,plays,distinct_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


ORDER BY played_on ASC works because date SQL lite recognizes dates as YYY-MM-DD. Interestingly, SQLite interestingly uses ISO date format which sorts dates alphabetically. Even though it is displayed as numbers, ISO is designed to make the numerical and string representation of date agree.

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [10]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

I had the most trouble on question 5, specifically in figuring out the rounding feature to get the answer to one decimal. When I originally tried 60 instead of 60.0, it output 19.0 for the top row. Another attempt where I removed the 1 after 60.0 output 20.0. I did the arithmatic on my own and figured 19.6 was the most correct answer, so I went with 60.0, 1. I now realize that running 60 removes the decimal from the answer before it is rounded.